In [7]:
import pandas as pd

# Carrega o arquivo JSON para um DataFrame
df = pd.read_json('../data/raw/tv.ibyte.json')

df.head()

False
True


,microdata,json-ld
0,"[{'@type': 'BreadcrumbList', '@context': 'http...",[]
1,"[{'@type': 'BreadcrumbList', '@context': 'http...",[]
2,"[{'@type': 'BreadcrumbList', '@context': 'http...",[]
3,"[{'@type': 'BreadcrumbList', '@context': 'http...",[]
4,"[{'@type': 'BreadcrumbList', '@context': 'http...",[]


TAGS

EXPLICAR QUE O PROCESSAMENTO VAI SER DA URL NAO DO TITULO

PRÉ PROCESSAMENTO PARA EXTRAIR INFORMAÇÕES DA URL



### Atividade 4 - NER - Ciências de Dados
### Professor André Camara
#### Alunos Diego Delgado, Laís de Jesus, Tiago

A base de dados escolhida foi os dados de Televisão (tv.ibyte.json)


Adicionar explicação de porque fez o tratamento de dados para buscar url
Explicar que usou o gemini para fazer anotações


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import json
import re
import os
from urllib.parse import unquote


def limpar_slug_url(url: str) -> str:
    """Extrai o slug da URL do produto e o converte em um título legível."""
    if not url:
        return ""

    # Remove parâmetros de busca (query params) ou barras no final, se houver
    url_limpa = url.split("?")[0].rstrip("/")

    # Pega o último segmento da URL (onde fica o nome comercial do produto)
    slug = url_limpa.split("/")[-1]

    # Remove o sufixo "/p" ou de paginação de e-commerce caso venha junto
    if slug == "p":
        slug = url_limpa.split("/")[-2]

    # Decodifica caracteres especiais da URL (ex: %20 -> espaço)
    texto_decodificado = unquote(slug)

    # Substitui hífenes e traços por espaços
    titulo = texto_decodificado.replace("-", " ")

    # Remove múltiplos espaços consecutivos e limpa as pontas
    titulo = re.sub(r"\s+", " ", titulo).strip()

    return titulo


def processar_json_file(caminho_entrada: str, caminho_saida: str):
    titulos_extraidos = []

    with open(caminho_entrada, "r", encoding="utf-8") as f:
        data = json.load(f)  # Load the entire JSON file

    # The file is expected to be a list of JSON objects
    if isinstance(data, list):
        for item in data:  # Iterate through each item in the loaded list
            # Each 'item' should now be a dictionary
            if isinstance(item, dict):
                microdata_list = item.get("microdata", [])

                for md_entry in microdata_list:
                    # Procura a entidade do tipo "Product"
                    if isinstance(md_entry, dict) and md_entry.get("@type") == "Product":
                        raw_url = md_entry.get("url", "")
                        titulo = limpar_slug_url(raw_url)

                        if titulo:
                            titulos_extraidos.append(titulo)
            else:
                print(f"Warning: Expected a dictionary in the JSON array, but found type {type(item)}. Skipping: {item}")
    elif isinstance(data, dict):
        # Handle case if the root is a single dictionary, not a list, but has 'microdata'
        microdata_list = data.get("microdata", [])
        for md_entry in microdata_list:
            if isinstance(md_entry, dict) and md_entry.get("@type") == "Product":
                raw_url = md_entry.get("url", "")
                titulo = limpar_slug_url(raw_url)
                if titulo:
                    titulos_extraidos.append(titulo)
    else:
        print(f"Error: The JSON file root is neither a list nor a dictionary. Found type: {type(data)}. Cannot process.")
        return

    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(caminho_saida), exist_ok=True)

    # Salva os títulos limpos em um arquivo de texto (um por linha)
    with open(caminho_saida, "w", encoding="utf-8") as file_out:
        for t in titulos_extraidos:
            file_out.write(t + "\n")

    print(
        f"Sucesso! {len(titulos_extraidos)} títulos foram extraídos e salvos em: {caminho_saida}"
    )


# --- Execução ---
if __name__ == "__main__":
    # Altere os caminhos de acordo com a sua pasta local
    ARQUIVO_ENTRADA = "../data/raw/tv.ibyte.json"  # substitua pelo nome exato do seu arquivo na pasta raw
    ARQUIVO_SAIDA = "../data/processed/titulos_tv_limpos.txt"

    processar_json_file(ARQUIVO_ENTRADA, ARQUIVO_SAIDA)


Sucesso! 818 títulos foram extraídos e salvos em: ../data/processed/titulos_tv_limpos.txt


Script para Anotações

In [13]:
import json
import os
import re

PATTERNS = {
    "MARCA": [
        r"\b(samsung|lg|toshiba|philco|tcl|sony|panasonic|multilaser|aoc|semp|goldentec|apple|amazon|jbl|hq|keo|intelbras|elg|brasforma|exbom|loctek|waleu|trust)\b"
    ],
    "TIPO": [
        r"\b(smart tv|tv monitor|monitor tv|televisao smart|televisao|fire tv amazon stick|fire tv stick|smart tv box|apple tv|projetor smart|projetor|soundbar|sound bar|home theater|conversor e gravador digital|conversor digital|conversor|antena digital|antena interna|antena externa|antena|suporte articulado|suporte fixo|suporte triarticulado|suporte de mesa|suporte de parede|suporte para monitor|suporte|controle remoto universal|controle remoto|caixa de som|roku express|chromecast|tv|tela)\b"
    ],
    "TELA_TAMANHO": [
        r"\b(100|86|85|82|77|75|71|70|65|60|58|56|55|50|49|48|43|42|40|39|32|28|26|24|23\s*6|20|17|14|13|10)\s*(polegadas|pol|\b)"
    ],
    "RESOLUCAO": [
        r"\b(ultra hd 4k|ultra hd 8k|ultra hd|4k uhd|8k|4k|fhd|full hd|hdtv|hd)\b"
    ],
    "TECNOLOGIA": [
        r"\b(neo\s*qled|neoqled|qled|oled evo|oled|mini led|dled|nanocell|qned|quantum dot|ips|led|lcd)\b"
    ],
    "MODELO": [
        r"\b([a-z]{1,4}\d{2,5}[a-z0-9]*|\d{2}[a-z]{1,4}\d{2,5}[a-z0-9]*|the frame|the sero|crystal uhd|crystal)\b"
    ],
}


def extrair_entidades(texto: str):
    matches = []
    for label, regex_list in PATTERNS.items():
        for reg in regex_list:
            for m in re.finditer(reg, texto, flags=re.IGNORECASE):
                start, end = m.span()
                matches.append((start, end, label))

    # Ordena pelo início e prioriza os spans mais longos
    matches.sort(key=lambda x: (x[0], -(x[1] - x[0])))

    entidades_finais = []
    ultimo_fim = -1
    for start, end, label in matches:
        if start >= ultimo_fim:
            entidades_finais.append([start, end, label])
            ultimo_fim = end

    return entidades_finais


# Caminhos dos arquivos
arquivo_entrada = "../data/processed/titulos_tv_limpos.txt"
arquivo_saida = "../data/annotations/tv.ibyte.jsonl"

os.makedirs(os.path.dirname(arquivo_saida), exist_ok=True)

linhas_processadas = 0
with (
    open(arquivo_entrada, "r", encoding="utf-8") as f_in,
    open(arquivo_saida, "w", encoding="utf-8") as f_out,
):
    for linha in f_in:
        texto = linha.strip()
        if not texto:
            continue
        entidades = extrair_entidades(texto)
        registro = {"text": texto, "entities": entidades}
        f_out.write(json.dumps(registro, ensure_ascii=False) + "\n")
        linhas_processadas += 1

print(
    f"Pronto! Todas as {linhas_processadas} linhas foram anotadas em {arquivo_saida}."
)

Pronto! Todas as 818 linhas foram anotadas em ../data/annotations/tv.ibyte.jsonl.
